# Propagation of Errors · Ultrasound Beamforming · Wave Physics
Measurement uncertainty → acoustic phased arrays → laser in concrete.

In [ ]:
from sympy import *
from IPython.display import display, Markdown

init_printing(use_latex='mathjax')

def step(label, expr=None):
    display(Markdown(f'**{label}**'))
    if expr is not None:
        display(expr)

def section(title):
    display(Markdown(f'---\n## {title}'))

## §1 — Propagation of Errors: The General Formula

In [ ]:
section('Propagation of Errors')

display(Markdown(r'''
If $f = f(x_1, x_2, \ldots, x_n)$ and each $x_i$ has uncertainty $\sigma_i$:
$$\sigma_f^2 = \sum_{i=1}^n \left(\frac{\partial f}{\partial x_i}\right)^2 \sigma_i^2
+ 2\sum_{i<j} \frac{\partial f}{\partial x_i}\frac{\partial f}{\partial x_j}\,\text{Cov}(x_i, x_j)$$
For **independent** measurements: covariance terms vanish.
'''))

x, y, z = symbols('x y z', positive=True)
sx, sy, sz = symbols('sigma_x sigma_y sigma_z', positive=True)

def propagate(f, variables, sigmas):
    """Symbolic error propagation for independent measurements."""
    var_f = sum((diff(f, v)**2 * s**2) for v, s in zip(variables, sigmas))
    return sqrt(var_f)

# Example 1: f = x + y
f1 = x + y
sf1 = propagate(f1, [x,y], [sx,sy])
step('f = x + y:', f1)
step('σ_f =', sf1)
step('Simplified =', simplify(sf1))

# Example 2: f = x * y
f2 = x * y
sf2 = propagate(f2, [x,y], [sx,sy])
step('f = x·y:', f2)
step('σ_f =', simplify(sf2))
step('Relative error: σ_f/f =', simplify(sf2/f2))
step('= √[(σ_x/x)² + (σ_y/y)²]  — add relative errors in quadrature')

# Example 3: power law f = x^n
n_exp = symbols('n', positive=True)
f3 = x**n_exp
sf3 = propagate(f3, [x], [sx])
step('f = xⁿ:', f3)
step('σ_f =', simplify(sf3))
step('Relative: σ_f/f = n·σ_x/x  — exponent multiplies relative error')

## §2 — Error Propagation: Physical Examples

In [ ]:
section('Physical Error Examples')

# Kinetic energy KE = ½mv²
m, v_sym = symbols('m v', positive=True)
sm, sv = symbols('sigma_m sigma_v', positive=True)
KE = Rational(1,2)*m*v_sym**2
sKE = propagate(KE, [m, v_sym], [sm, sv])
step('KE = ½mv²:', KE)
step('σ_KE =', simplify(sKE))
step('Relative σ_KE/KE =', simplify(sKE/KE))
step('= √[(σ_m/m)² + (2σ_v/v)²]  — velocity error counts double')

# Speed of sound: v = λf
lam, f_sym = symbols('lambda f', positive=True)
sl, sf_s = symbols('sigma_lambda sigma_f', positive=True)
v_sound = lam * f_sym
sv_s = propagate(v_sound, [lam, f_sym], [sl, sf_s])
step('\nv = λf (speed of sound from wavelength and frequency):', v_sound)
step('σ_v =', simplify(sv_s))

# Snell's law angle uncertainty
n1, n2, theta1 = symbols('n_1 n_2 theta_1', positive=True)
st1 = symbols('sigma_theta1', positive=True)
theta2 = asin(n1/n2 * sin(theta1))
step('\nSnell refraction angle θ₂ = arcsin(n₁/n₂ · sin θ₁):', theta2)
st2 = propagate(theta2, [theta1], [st1])
step('σ_θ₂ =', simplify(st2))
step('Diverges as θ₂ → 90° (total internal reflection) — critical angle measurement is imprecise')

## §3 — Wave Equation and Dispersion Relations

In [ ]:
section('Wave Equation')

x, t, c_w = symbols('x t c', positive=True)
u = Function('u')(x, t)

wave_eq = Eq(u.diff(t,2), c_w**2 * u.diff(x,2))
step('Wave equation:', wave_eq)

sol = dsolve(wave_eq)
step('General solution (d\'Alembert):', sol)
step('f(x−ct): rightward wave,  g(x+ct): leftward wave')

# Plane wave ansatz
k, omega = symbols('k omega', real=True)
display(Markdown(r'''
**Plane wave ansatz** $u = e^{i(kx - \omega t)}$:
$$(-\omega^2) = c^2(-k^2) \implies \omega = \pm ck$$
'''))
disp_linear = Eq(omega, c_w*k)
step('Linear dispersion relation:', disp_linear)
step('Phase velocity v_p = ω/k =', c_w)
step('Group velocity v_g = dω/dk =', diff(c_w*k, k))
step('v_p = v_g = c  →  non-dispersive (all frequencies travel same speed)')

display(Markdown(r'''
| Medium | Dispersion relation | Dispersive? |
|---|---|---|
| Vacuum (EM) | $\omega = ck$ | No |
| Shallow water | $\omega = \sqrt{gh}\,k$ | No |
| Deep water | $\omega = \sqrt{gk}$ | **Yes** |
| Optical fiber | $\omega = c/n(\omega)\cdot k$ | **Yes** |
| Quantum particle | $\omega = \hbar k^2/2m$ | **Yes** |
| Concrete (P-wave) | $\omega \approx c_p k$ (with attenuation) | Weakly |
'''))

## §4 — Ultrasound Beamforming: Delay and Sum

In [ ]:
section('Ultrasound Phased Array Beamforming')

display(Markdown(r'''
An $N$-element array steers a beam by applying **time delays** $\tau_n$ to each element.
Elements spaced $d$ apart. Target at angle $\theta$ from broadside.

```
Element:  0    1    2    3  ...  N-1
          |    |    |    |
          d    d    d    d
               θ ↗  (beam direction)
```
'''))

d_arr, theta_b, c_s, n_elem = symbols('d theta c n', positive=True)
lam_us = symbols('lambda', positive=True)

# Delay for element n to steer beam to angle theta
tau_n = n_elem * d_arr * sin(theta_b) / c_s
step('Time delay for element n: τ_n = n·d·sin(θ)/c =', tau_n)

# Array factor (far field)
display(Markdown(r'''
**Array factor** $AF(\theta)$ — interference pattern of $N$ elements:
$$AF(\theta) = \sum_{n=0}^{N-1} e^{i n k d (\sin\theta - \sin\theta_0)}$$

This is a geometric series. Let $\psi = kd(\sin\theta - \sin\theta_0)$:
'''))

N_sym, psi = symbols('N psi', positive=True)
AF = (1 - exp(I*N_sym*psi)) / (1 - exp(I*psi))
step('Array factor (geometric sum):', AF)

# Magnitude squared
display(Markdown(r'''
$$|AF(\psi)|^2 = \frac{\sin^2(N\psi/2)}{\sin^2(\psi/2)}$$

**Main lobe** at $\psi=0$ (i.e. $\theta=\theta_0$): $|AF|^2_{max} = N^2$  
**First null** at $\psi = 2\pi/N$  
**3dB beamwidth** $\approx 0.886\lambda/(Nd)$ (radians)
'''))

BW = Rational(886,1000)*lam_us / (N_sym * d_arr)
step('3dB beamwidth ≈ 0.886λ/(Nd) =', BW)

# Numeric: 64-element, 5 MHz, d=λ/2
c_tissue = 1540.0  # m/s
f_us = 5e6         # Hz
lam_num = c_tissue / f_us
N_num = 64
d_num = lam_num / 2
BW_num = 0.886 * lam_num / (N_num * d_num)
step(f'64-element, 5 MHz, d=λ/2:')
print(f'  λ = {lam_num*1e3:.3f} mm')
print(f'  d = {d_num*1e3:.3f} mm')
print(f'  Beamwidth = {degrees(BW_num):.2f}°')
print(f'  At 50mm depth: lateral resolution = {50e-3*BW_num*1e3:.2f} mm')

## §5 — Acoustic Wave in Solids: P-waves and S-waves

In [ ]:
section('Acoustic Waves in Solids')

E_mod, nu, rho = symbols('E nu rho', positive=True)

# P-wave (longitudinal) speed
c_p = sqrt(E_mod*(1-nu) / (rho*(1+nu)*(1-2*nu)))
step('P-wave (longitudinal) speed c_p =', c_p)

# S-wave (shear) speed
G_mod = E_mod / (2*(1+nu))
c_s_solid = sqrt(G_mod / rho)
step('Shear modulus G = E/[2(1+ν)] =', G_mod)
step('S-wave (shear) speed c_s =', c_s_solid)

# Ratio
ratio = simplify(c_p / c_s_solid)
step('c_p/c_s =', ratio)
step('For ν=0.3 (concrete): c_p/c_s =', float(ratio.subs(nu, 0.3)))

display(Markdown(r'''
| Material | $c_p$ (m/s) | $c_s$ (m/s) | $\nu$ |
|---|---|---|---|
| Air | 343 | — | — |
| Water | 1480 | — | 0.5 |
| Soft tissue | 1540 | ~10 | ~0.5 |
| Concrete | 3500–4500 | 2000–2500 | 0.20 |
| Steel | 5960 | 3260 | 0.29 |
| Fused silica | 5970 | 3760 | 0.17 |
'''))

## §6 — Laser Ultrasonics in Concrete

In [ ]:
section('Laser Ultrasonics: Non-Contact NDT of Concrete')

display(Markdown(r'''
**Laser ultrasound** (LUS): pulsed laser generates thermoelastic stress wave;
interferometer (vibrometer) detects surface displacement — fully non-contact.

```
Nd:YAG pulse (10 ns, 532 nm)         Detection laser (CW, 633 nm)
       ↓                                    ↑
  [CONCRETE SURFACE]  ──── P-wave ────►  defect/rebar
                      ◄─── reflection ──
```

**Generation mechanisms**:
- Thermoelastic (below ablation): laser heats surface → thermal expansion → stress wave
- Ablation (high fluence): plasma recoil → stronger but damages surface
'''))

# Thermoelastic source strength
alpha_T, E_laser, kappa, C_p_mat = symbols('alpha_T E_laser kappa C_p', positive=True)
# Characteristic displacement amplitude ~ α_T · E_laser / (ρ c_p)
u_thermo = alpha_T * E_laser / (rho * C_p_mat)
step('Thermoelastic displacement ~ α_T · E/(ρ C_p) =', u_thermo)

# Time of flight → depth
t_tof, c_concrete = symbols('t_TOF c_p', positive=True)
depth = c_concrete * t_tof / 2
step('Defect depth from time of flight: d = c_p · t_TOF / 2 =', depth)

# Depth resolution
tau_pulse = symbols('tau_pulse', positive=True)
delta_d = c_concrete * tau_pulse / 2
step('Depth resolution Δd = c_p · τ_pulse / 2 =', delta_d)

# Numeric: 10 ns pulse, c_p = 4000 m/s
print(f'10 ns pulse, c_p = 4000 m/s:')
print(f'  Depth resolution = {4000*10e-9/2*1e3:.2f} mm')
print(f'  Rebar at 50 mm: t_TOF = {2*50e-3/4000*1e6:.1f} μs')

display(Markdown(r'''
**Applications**:
- Rebar location and corrosion state
- Delamination and void detection
- Bridge deck, dam, nuclear containment inspection
- Coupling with SAFT (synthetic aperture focusing) for 3D imaging
'''))

## §7 — Reaction Rates: Chemistry Kinetics

In [ ]:
section('Reaction Rates and Kinetics')

t, k_rate, A_conc, Ea, R_gas, T_temp = symbols('t k A E_a R T', positive=True)
n_order = symbols('n', positive=True)

# nth order rate law
rate = k_rate * A_conc**n_order
step('Rate law: r = k[A]ⁿ =', rate)

# First order ODE: d[A]/dt = -k[A]
A_func = Function('A')(t)
ode1 = Eq(A_func.diff(t), -k_rate * A_func)
step('First order ODE:', ode1)
sol1 = dsolve(ode1, A_func)
step('Solution:', sol1)
step('→ [A](t) = [A]₀ · e^(−kt)')

# Half life
t_half = log(2) / k_rate
step('Half-life t½ = ln2/k =', t_half)

# Arrhenius
A_pre = symbols('A_0', positive=True)
k_arr = A_pre * exp(-Ea / (R_gas * T_temp))
step('Arrhenius: k(T) = A·exp(−Ea/RT) =', k_arr)

# Linearize: ln k = ln A - Ea/RT
step('Linearized: ln(k) =', log(k_arr).expand())
step('Plot ln(k) vs 1/T: slope = −Ea/R  →  extract activation energy')

# Error in Ea from slope measurement
m_slope, sm_slope = symbols('m sigma_m', real=True)
Ea_from_slope = -R_gas * m_slope
sEa = propagate(Ea_from_slope, [m_slope], [sm_slope])
step('σ_Ea from slope uncertainty:', simplify(sEa))
step('= R · σ_m  →  error in Ea proportional to error in measured slope')

## §8 — Connecting Errors to Measurement: SNR and Fisher Information

In [ ]:
section('Fisher Information and Cramér-Rao Bound')

display(Markdown(r'''
**Cramér-Rao bound**: no unbiased estimator can achieve variance below
$$\text{Var}(\hat{\theta}) \geq \frac{1}{I(\theta)}$$
where $I(\theta)$ is the **Fisher information**:
$$I(\theta) = -\mathbb{E}\left[\frac{\partial^2}{\partial\theta^2}\ln p(x|\theta)\right]$$

This is the **fundamental limit** on measurement precision — connects
error propagation to information theory.
'''))

# Gaussian likelihood: Fisher info = 1/σ²
theta, sigma_n, x_meas = symbols('theta sigma x', real=True)
log_p = -Rational(1,2)*((x_meas - theta)/sigma_n)**2 - log(sigma_n)
step('ln p(x|θ) for Gaussian:', log_p)
d2_log_p = diff(log_p, theta, 2)
step('d²ln p/dθ² =', d2_log_p)
I_fisher = -d2_log_p
step('Fisher information I(θ) = 1/σ² =', I_fisher)
step('CR bound: Var(θ̂) ≥ σ²  →  Gaussian is efficient (achieves the bound)')

display(Markdown(r'''
**For beamforming / phase retrieval**:
- Fisher info increases with SNR → more photons/signal = tighter phase estimate
- GS algorithm approaches CR bound as number of measurements increases
- Dispersive diversity ($|\beta_2 z|$ large) → higher Fisher info for phase
'''))